In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [2]:
# set device to cuda if it is available, else use cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
with open("anna.txt", "r") as file:
    content = file.read()

#### Textual Preprocessing for vanilla RNNs

In [5]:
import re

def clean_text(text):
    # convert into lower case letter
    text = text.lower()
    # 2. Add spaces around punctuation marks so they become separate tokens
    text = re.sub(r"([.,!?\"():;])", r" \1 ", text)
    # collapse multiple spaces into a single space
    text = re.sub(r"\s+", " ", text)
    tokens = text.strip().split()
    return tokens

tokens = clean_text(content)

In [6]:
unique_words = sorted(list(set(tokens)))
vocab_size = len(unique_words)

word_to_idx = {word : idx for idx, word in enumerate(unique_words)}
idx_to_word = {idx : word for idx, word in enumerate(unique_words)}

print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 14972


#### Step 2: Creating Sequences (The Dataset)
RNNs learn by looking at a sequence of words to predict the next one. If our sequence length is 3, the model looks at words `[1, 2, 3]` to predict word `[4]`.

In [7]:
sequence_length = 4

class TextDataset(Dataset):
    def __init__(self, words, word_to_idx, seq_length):
        self.words = words
        self.word_to_idx = word_to_idx
        self.seq_length  = seq_length
    def __len__(self):
        return len(self.words) - self.seq_length
    def __getitem__(self, index):
        # Get the sequence of words
        seq = self.words[index : index + self.seq_length]
        target = self.words[index + self.seq_length]
        # convert to indices
        seq_ix = [self.word_to_idx[w] for w in seq]
        target_ix = self.word_to_idx[target]
        return torch.tensor(seq_ix), torch.tensor(target_ix)

dataset = TextDataset(tokens, word_to_idx, sequence_length)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

### Step 3: Defining the Vanilla RNN Model
Our model needs three layers: <br>

- Embedding Layer: Converts word indices into dense mathematical vectors.

- RNN Layer: The core memory unit that processes the sequence sequentially.

- Linear Layer: Decodes the RNN output into a probability distribution over our entire vocabulary.

In [8]:
embedding_dim = 16
hidden_dim = 32

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(VanillaRNN, self).__init__()
        self.hidden_dim = hidden_dim

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # create Vanilla RNN layer (batch_first = True)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # output layer
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        batch_size = x.size(0)

        embeds = self.embedding(x)

        h0 = torch.zeros(1, batch_size, self.hidden_dim).to(x.device)

        out, _ = self.rnn(embeds, h0)

        out = out[:, -1, :]
        out = self.fc(out)
        return out


model = VanillaRNN(vocab_size, embedding_dim, hidden_dim)

In [10]:
model = model.to(device)

In [9]:
import torch

if torch.cuda.is_available():
    # Get basic device properties
    device_id = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_id)
    
    # Calculate memory metrics in Gigabytes (GB)
    total_mem = props.total_memory / (1024 ** 3)
    allocated_mem = torch.cuda.memory_allocated(device_id) / (1024 ** 3)
    cached_mem = torch.cuda.memory_reserved(device_id) / (1024 ** 3)
    free_mem = total_mem - allocated_mem
    
    print(f"📡 GPU Device Name: {props.name}")
    print(f"📊 Total VRAM Capacity: {total_mem:.2f} GB")
    print(f"🔴 Currently Used by PyTorch: {allocated_mem:.2f} GB")
    print(f"🟡 Reserved/Cached Memory: {cached_mem:.2f} GB")
    print(f"🟢 Free VRAM Available: {free_mem:.2f} GB")
else:
    print("❌ CUDA is not available. System is running on CPU.")


📡 GPU Device Name: NVIDIA T1200 Laptop GPU
📊 Total VRAM Capacity: 4.00 GB
🔴 Currently Used by PyTorch: 0.00 GB
🟡 Reserved/Cached Memory: 0.00 GB
🟢 Free VRAM Available: 4.00 GB


##### LSTM Model Architecture

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        batch_size = x.size(0)

        embeds = self.embedding(x)

        h0 = torch.zeros(1, batch_size, self.hidden_dim).to(x.device)
        c0 = torch.zeros(1, batch_size, self.hidden_dim).to(x.device)

        out, _ = self.lstm(embeds, (h0, c0))

        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [13]:
model = LSTMModel(vocab_size, embedding_dim, hidden_dim)
model = model.to(device)

In [14]:
from tqdm import tqdm

epochs = 100
learning_rate = 0.01

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in tqdm(range(epochs)):
    model.train()
    total_loss = 0
    
    for seq_batch, target_batch in dataloader:
        seq_batch, target_batch = seq_batch.to(device), target_batch.to(device)
        # 1. Clear gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        predictions = model(seq_batch)
        
        # 3. Calculate loss
        loss = criterion(predictions, target_batch)
        
        # 4. Backward pass (calculate gradients)
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        total_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

 10%|█         | 10/100 [19:01<2:30:04, 100.05s/it]

Epoch 10/100 | Loss: 4.7031


 20%|██        | 20/100 [9:53:35<10:13:53, 460.41s/it]   

Epoch 20/100 | Loss: 4.7554


 30%|███       | 30/100 [10:01:23<1:08:35, 58.80s/it] 

...

Epoch 80/100 | Loss: 5.2643


 90%|█████████ | 90/100 [10:43:39<06:45, 40.51s/it]

Epoch 90/100 | Loss: 5.2430


100%|██████████| 100/100 [10:50:16<00:00, 390.16s/it]

Epoch 100/100 | Loss: 5.2650


#### Step 5: Text Generation (Inference)
To generate text, we feed the model a "seed" sequence, ask it to predict the next word, append that word to our sequence, and repeat the process.

In [15]:
loss.item()

4.5622878074646

In [16]:
def generate_text(model, seed_text, num_words, word_to_ix, ix_to_word, seq_length):
    model.eval() # Set to evaluation mode
    
    # Process seed text
    words = seed_text.lower().split()
    
    with torch.no_grad():
        for _ in range(num_words):
            # Pad or truncate seed text to match sequence_length
            current_seq = words[-seq_length:]
            if len(current_seq) < seq_length:
                # If seed is too short, we'd normally pad it. 
                # For simplicity here, we assume seed is at least seq_length long.
                print("Seed text too short!")
                return
            
            # Convert to indices and tensor
            seq_ix = [word_to_ix[w] for w in current_seq]
            x = torch.tensor(seq_ix).unsqueeze(0) # Add batch dimension
            
            # Predict
            output = model(x.to(device))
            
            # Get the index of the word with highest probability
            predicted_ix = torch.argmax(output, dim=1).item()
            predicted_word = ix_to_word[predicted_ix]
            
            # Append to our generated list
            words.append(predicted_word)
            
    return ' '.join(words)

# Let's test it!
seed = "The wife had discovered"
generated_sentence = generate_text(model, seed, num_words=5, 
                                   word_to_ix=word_to_idx, 
                                   ix_to_word=idx_to_word, 
                                   seq_length=sequence_length)

print("\nGenerated Text:")
print(generated_sentence)


Generated Text:
the wife had discovered , " " " "


In [19]:
torch.save(model.state_dict(), "next_word_prediction_model.pth")

#### Using LSTM architecture
